# HormuzWatch ML Service — Colab + ngrok Deployment

Runs the FastAPI **anomaly-detection** service on Google Colab and exposes it to the public internet through an **ngrok** tunnel, so your Go backend (running elsewhere) can reach it as `ML_SERVICE_URL`.

## Prerequisites
1. Upload the `ml-service` folder (or `git clone` your repo) so that `app.py` sits at `/content/ml-service/app.py`.
2. Add your **ngrok authtoken** to Colab **Secrets** with key `NGROK_AUTH_TOKEN` (or paste it when prompted).
3. Run every cell top-to-bottom.
4. Copy the printed `https://xxxx.ngrok.io` URL and set `ML_SERVICE_URL` on your Go backend to it.


In [ ]:
import os, subprocess, sys, time

# Where the ml-service code lives inside this Colab runtime.
# If you cloned the whole repo instead of just the folder, change this
# to e.g. '/content/HormuzWatch/ml-service'.
ML_DIR = '/content/ml-service'
os.makedirs(ML_DIR, exist_ok=True)
os.chdir(ML_DIR)

# --- OPTIONAL: upload the folder instead of cloning ---
# from google.colab import files
# uploaded = files.upload()  # select the ml-service folder .zip, then unzip below
# !unzip -oq ml-service.zip && rm ml-service.zip

print('Working dir:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))


In [ ]:
# Install runtime dependencies + pyngrok (pyngrok is not in requirements.txt)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
print('Dependencies installed.')


In [ ]:
# --- Configure ngrok authtoken ---
NGROK_TOKEN = None
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
except Exception:
    pass

if not NGROK_TOKEN:
    NGROK_TOKEN = input('Paste your ngrok authtoken: ').strip()

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('ngrok authtoken configured.')


In [ ]:
# --- Launch the ML service in the background ---
PORT = int(os.environ.get('PORT', '8090'))
# CORS defaults to '*' in app.py; that is intentional for the ephemeral
# Colab/ngrok tunnel. Override with a real origin in production.
os.environ.setdefault('ALLOWED_ORIGINS', '*')
os.environ.setdefault('MODELS_DIR', './models')

launch_cmd = (
    f"nohup {sys.executable} -m uvicorn app:app "
    f"--host 0.0.0.0 --port {PORT} --timeout-keep-alive 120 > uvicorn.log 2>&1 &
)
print('Launching:', launch_cmd.strip())
os.system(launch_cmd)
time.sleep(6)
print('--- uvicorn.log (tail) ---')
print(subprocess.run(['tail', '-n', '25', 'uvicorn.log'], capture_output=True, text=True).stdout)


In [ ]:
# --- Open the public ngrok tunnel ---
public_url = ngrok.connect(PORT, bind_tls=True).public_url
print('ML service is now PUBLIC at:')
print(public_url)
print()
print('Set this on your Go backend (.env / deployment env):')
print(f'  ML_SERVICE_URL={public_url}')


## Wire up the Go backend

Your `server` calls the ML service at `ML_SERVICE_URL` + `/api/predict` with a **500 ms timeout** and **graceful fallback** to a local heuristic if the service is unreachable or slow. So even if Colab/ngrok latency exceeds 500 ms, scoring still works (it just won't use the model).

In `.env` (or your deployment environment) **on the Go server machine**:

```
ML_SERVICE_URL=https://xxxx.ngrok.io
```

Then restart the Go server. The 30s in-memory ML result cache in the Go client means a single slow Colab call only impacts the first request for a given track.

> ⚠️ **Colab runtimes are ephemeral.** When the runtime disconnects, the tunnel dies and the URL changes. Re-run this notebook to get a fresh URL and update `ML_SERVICE_URL` on the Go backend. For anything beyond a demo, host the ML service near the Go backend (Docker image / VM) and set `ALLOWED_ORIGINS` to your frontend origin(s) instead of `*`.


In [ ]:
# --- Sanity check: hit /health through the public tunnel ---
import urllib.request, json
try:
    with urllib.request.urlopen(public_url.rstrip('/') + '/health', timeout=15) as r:
        print('Health:', json.loads(r.read()))
except Exception as e:
    print('Health check failed:', repr(e))
